In [1]:
# ===== CELL 0: ENV FIX (run once) =====
!pip -q install -U --no-cache-dir \
"numpy==2.2.6" \
"opencv-python-headless==4.12.0.88" \
"timm==1.0.22" \
"kagglehub" \
"matplotlib"

import numpy as np, timm, torch
print("numpy:", np.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("timm:", timm.__version__)

print("\nIMPORTANT: Restart session/kernel now (Kaggle: Notebook -> Restart session), then run CELL 1.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 114.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 166.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 232.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 252.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 268.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 221.5 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
google-colab 1.0.0 requires jupyte

In [2]:
# ===== CELL 1: IMPROVED FULL PIPELINE =====
import os, gc, math, random, copy, json, zipfile, shutil
from pathlib import Path
from contextlib import nullcontext

import numpy as np
from PIL import Image, ImageFilter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import matplotlib.pyplot as plt
import cv2

# -------------------- Seed --------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(42)

# -------------------- AMP --------------------
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)

def autocast_or_noop():
    if USE_CUDA:
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return nullcontext()

# -------------------- Config --------------------
class Config:
    MODEL_CANDIDATES = [
        "swinv2_small_window8_256.ms_in1k",
        "swinv2_tiny_window8_256.ms_in1k",
        #"swin_base_patch4_window7_224",
        #"swin_small_patch4_window7_224",
    ]

    IMG_SIZE_STAGE12 = 224
    IMG_SIZE_STAGE3 = 256

    NUM_CLASSES = None
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    ACCUM_STEPS = 2

    LABEL_SMOOTHING_STAGE1 = 0.05
    LABEL_SMOOTHING_BALANCED = 0.02
    WEIGHT_DECAY = 0.03
    DROP_PATH_RATE = 0.15
    HEAD_DROPOUT = 0.20

    STAGE1_EPOCHS = 5
    STAGE2_MAX_EPOCHS = 15
    STAGE3_EPOCHS = 5

    MIN_DELTA_F1_STAGE2 = 1e-3
    PATIENCE_STAGE2 = 5
    MIN_EPOCHS_STAGE2 = 10

    MIN_DELTA_F1_STAGE3 = 5e-4
    PATIENCE_STAGE3 = 3
    MIN_EPOCHS_STAGE3 = 3

    STAGE1_BACKBONE_LR = 2e-5
    STAGE1_HEAD_LR = 7e-4

    STAGE2_LAYER3_LR = 8e-5
    STAGE2_LAYER2_LR = 4e-5
    STAGE2_HEAD_LR = 2.5e-4

    STAGE3_LR = 1.2e-5
    STAGE3_HEAD_LR = 2.5e-5

    CLIP_GRAD_NORM = 1.0

    OUT_DIR = Path("/kaggle/working")
    SAVE_PREFIX = "dermnet_swin_balanced"

print("Device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)

# -------------------- Data (DermNet) --------------------
DATA_ROOT = None
if Path("/kaggle/input/dermnet").exists():
    DATA_ROOT = Path("/kaggle/input/dermnet")
else:
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download("shubhamgoel27/dermnet"))

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
assert TRAIN_DIR.exists(), f"TRAIN_DIR not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"TEST_DIR not found: {TEST_DIR}"

IMG_EXT = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

def scan_split(root_dir: Path):
    class_names = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
    label_map = {c: i for i, c in enumerate(class_names)}

    paths, labels = [], []
    for c in class_names:
        cdir = root_dir / c
        for f in cdir.iterdir():
            if f.is_file() and f.suffix in IMG_EXT:
                paths.append(str(f))
                labels.append(label_map[c])

    return class_names, label_map, np.array(paths), np.array(labels, dtype=np.int64)

class_names, label_map, all_train_paths, all_train_labels = scan_split(TRAIN_DIR)
_, _, test_paths, test_labels = scan_split(TEST_DIR)

Config.NUM_CLASSES = len(class_names)
print("NUM_CLASSES:", Config.NUM_CLASSES, "| train:", len(all_train_paths), "| test:", len(test_paths))

# -------------------- Stratified split --------------------
def stratified_split(paths, labels, val_ratio=0.10, seed=42):
    rng = np.random.RandomState(seed)
    train_idx, val_idx = [], []
    num_classes = labels.max() + 1

    for c in range(num_classes):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        n_val = max(1, int(len(idx) * val_ratio))
        val_idx.append(idx[:n_val])
        train_idx.append(idx[n_val:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx

train_idx, val_idx = stratified_split(all_train_paths, all_train_labels, val_ratio=0.10, seed=42)
train_paths, train_labels = all_train_paths[train_idx], all_train_labels[train_idx]
val_paths, val_labels = all_train_paths[val_idx], all_train_labels[val_idx]

print("split -> train:", len(train_paths), "| val:", len(val_paths))

train_counts = np.bincount(train_labels, minlength=Config.NUM_CLASSES)
print("train class count min/max:", int(train_counts.min()), int(train_counts.max()))

# -------------------- Preprocessing: color constancy + CLAHE + NEW unsharp mask --------------------
class ShadesOfGrayColorConstancy:
    def __init__(self, p=6, eps=1e-6):
        self.p = p
        self.eps = eps

    def __call__(self, img: Image.Image) -> Image.Image:
        x = np.asarray(img).astype(np.float32) / 255.0
        rgb = np.power(np.mean(np.power(x, self.p), axis=(0, 1)), 1.0 / self.p)
        scale = np.mean(rgb) / (rgb + self.eps)
        x = np.clip(x * scale[None, None, :], 0.0, 1.0)
        return Image.fromarray((x * 255.0).astype(np.uint8))

class CLAHE_LAB:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img: Image.Image) -> Image.Image:
        x = np.asarray(img).astype(np.uint8)
        lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l2 = self.clahe.apply(l)
        lab2 = cv2.merge([l2, a, b])
        rgb2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
        return Image.fromarray(rgb2)

class UnsharpMaskRGB:
    def __init__(self, radius=1.2, percent=125, threshold=2):
        self.radius = radius
        self.percent = percent
        self.threshold = threshold

    def __call__(self, img: Image.Image) -> Image.Image:
        return img.filter(
            ImageFilter.UnsharpMask(
                radius=self.radius,
                percent=self.percent,
                threshold=self.threshold
            )
        )

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def build_transforms(img_size: int, train: bool):
    base_enh = [
        ShadesOfGrayColorConstancy(p=6),
        CLAHE_LAB(clip_limit=2.0, tile_grid_size=(8, 8)),
        UnsharpMaskRGB(radius=1.2, percent=125, threshold=2),
    ]

    if train:
        return transforms.Compose(base_enh + [
            transforms.RandomResizedCrop(img_size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=8),
            transforms.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.05, hue=0.01),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    else:
        return transforms.Compose(base_enh + [
            transforms.Resize(img_size + 32),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])

# -------------------- Dataset --------------------
class SkinDataset(Dataset):
    def __init__(self, paths, labels, transform=None, max_retries=10):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        for _ in range(self.max_retries):
            p = self.paths[idx]
            y = int(self.labels[idx])
            try:
                img = Image.open(p).convert("RGB")
                if self.transform:
                    img = self.transform(img)
                return img, torch.tensor(y, dtype=torch.long)
            except Exception:
                idx = random.randint(0, len(self.paths) - 1)
        raise RuntimeError(f"Too many failed loads. last={p}")

# -------------------- Loaders --------------------
def build_loaders(img_size: int):
    tr_tf = build_transforms(img_size, train=True)
    va_tf = build_transforms(img_size, train=False)

    tr_ds = SkinDataset(train_paths, train_labels, tr_tf)
    va_ds = SkinDataset(val_paths, val_labels, va_tf)
    te_ds = SkinDataset(test_paths, test_labels, va_tf)

    common_train_kwargs = dict(
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=USE_CUDA,
    )
    common_eval_kwargs = dict(
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=USE_CUDA,
    )

    if Config.NUM_WORKERS > 0:
        common_train_kwargs["persistent_workers"] = True
        common_eval_kwargs["persistent_workers"] = True

    tr_loader = DataLoader(tr_ds, **common_train_kwargs)
    va_loader = DataLoader(va_ds, **common_eval_kwargs)
    te_loader = DataLoader(te_ds, **common_eval_kwargs)
    return tr_loader, va_loader, te_loader

train_loader_224, val_loader_224, _ = build_loaders(Config.IMG_SIZE_STAGE12)

# -------------------- Light mixup only for warmup --------------------
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy

def build_mixup(num_classes):
    return Mixup(
        mixup_alpha=0.10,
        cutmix_alpha=0.0,
        prob=0.25,
        switch_prob=0.0,
        mode="batch",
        label_smoothing=Config.LABEL_SMOOTHING_STAGE1,
        num_classes=num_classes,
    )

mixup_fn = build_mixup(Config.NUM_CLASSES)
ce_soft = SoftTargetCrossEntropy()
ce_eval = nn.CrossEntropyLoss()

# -------------------- Balanced Softmax Loss --------------------
class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, samples_per_class, label_smoothing=0.0):
        super().__init__()
        samples_per_class = torch.tensor(samples_per_class, dtype=torch.float32)
        priors = samples_per_class / samples_per_class.sum()
        self.register_buffer("log_prior", torch.log(priors.clamp_min(1e-12)))
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        logits = logits + self.log_prior.unsqueeze(0)
        return F.cross_entropy(
            logits,
            target,
            label_smoothing=self.label_smoothing
        )

balanced_loss_stage2 = BalancedSoftmaxLoss(
    train_counts,
    label_smoothing=Config.LABEL_SMOOTHING_BALANCED
).to(DEVICE)

balanced_loss_stage3 = BalancedSoftmaxLoss(
    train_counts,
    label_smoothing=0.0
).to(DEVICE)

# -------------------- Model selection + size fix --------------------
def pick_model_name(candidates):
    avail = set(timm.list_models(pretrained=True))
    for name in candidates:
        if name in avail:
            return name
    return candidates[0]

MODEL_NAME = pick_model_name(Config.MODEL_CANDIDATES)
print("Chosen model:", MODEL_NAME)

def create_model(num_classes):
    m = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=Config.DROP_PATH_RATE,
    )

    if hasattr(m, "head") and isinstance(m.head, nn.Linear):
        in_f = m.head.in_features
        m.head = nn.Sequential(
            nn.Dropout(Config.HEAD_DROPOUT),
            nn.Linear(in_f, num_classes)
        )

    if hasattr(m, "set_input_size"):
        try:
            m.set_input_size(img_size=Config.IMG_SIZE_STAGE12)
        except Exception:
            pass

    if hasattr(m, "patch_embed"):
        if hasattr(m.patch_embed, "strict_img_size"):
            m.patch_embed.strict_img_size = False
        if hasattr(m.patch_embed, "img_size"):
            m.patch_embed.img_size = None

    return m

model = create_model(Config.NUM_CLASSES).to(DEVICE)

# -------------------- Trainable stages --------------------
def set_trainable_stage(model, stage: int):
    for p in model.parameters():
        p.requires_grad = False

    for n, p in model.named_parameters():
        if n.startswith("head."):
            p.requires_grad = True
        if "norm" in n:
            p.requires_grad = True

    if stage >= 2:
        for n, p in model.named_parameters():
            if n.startswith("layers.3."):
                p.requires_grad = True
            if n.startswith("layers.2."):
                p.requires_grad = True

    if stage >= 3:
        for p in model.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[Stage {stage}] Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M")

def build_optimizer_stage1(model):
    head_params, backbone_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith("head.") else backbone_params).append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": Config.STAGE1_BACKBONE_LR})
    if head_params:
        groups.append({"params": head_params, "lr": Config.STAGE1_HEAD_LR})

    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)

def build_optimizer_stage2(model):
    head, l3, l2, other = [], [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("head."):
            head.append(p)
        elif n.startswith("layers.3."):
            l3.append(p)
        elif n.startswith("layers.2."):
            l2.append(p)
        else:
            other.append(p)

    param_groups = []
    if other:
        param_groups.append({"params": other, "lr": Config.STAGE2_LAYER2_LR * 0.25})
    if l2:
        param_groups.append({"params": l2, "lr": Config.STAGE2_LAYER2_LR})
    if l3:
        param_groups.append({"params": l3, "lr": Config.STAGE2_LAYER3_LR})
    if head:
        param_groups.append({"params": head, "lr": Config.STAGE2_HEAD_LR})

    return optim.AdamW(param_groups, weight_decay=Config.WEIGHT_DECAY)

def build_optimizer_stage3(model):
    head_params, backbone_params = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (head_params if n.startswith("head.") else backbone_params).append(p)

    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": Config.STAGE3_LR})
    if head_params:
        groups.append({"params": head_params, "lr": Config.STAGE3_HEAD_LR})

    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)

# -------------------- Metrics --------------------
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def f1_from_cm(cm):
    eps = 1e-12
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)

    support = cm.sum(axis=1).astype(np.float64)
    f1_macro = np.mean(f1)
    f1_weighted = (f1 * support).sum() / (support.sum() + eps)
    return f1_macro, f1_weighted

# -------------------- Train / Eval loops --------------------
def train_one_epoch(model, loader, optimizer, criterion, mixup=None):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total, correct = 0, 0
    running_loss = 0.0

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    num_steps = len(loader)

    for step, (x, y) in enumerate(tqdm(loader, desc="TRAIN", leave=False), start=1):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        use_mix = (mixup is not None) and (x.size(0) % 2 == 0)
        if use_mix:
            x, y_soft = mixup(x, y)
            hard_y = y_soft.argmax(dim=1)
        else:
            y_soft = y
            hard_y = y

        with autocast_or_noop():
            logits = model(x)
            loss = criterion(logits, y_soft) / Config.ACCUM_STEPS

        if USE_CUDA:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % Config.ACCUM_STEPS == 0:
            if USE_CUDA:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, Config.CLIP_GRAD_NORM)

            if USE_CUDA:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        bs = hard_y.size(0)
        running_loss += loss.item() * bs * Config.ACCUM_STEPS
        pred = logits.argmax(dim=1)
        correct += (pred == hard_y).sum().item()
        total += bs

    if num_steps % Config.ACCUM_STEPS != 0:
        if USE_CUDA:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, Config.CLIP_GRAD_NORM)

        if USE_CUDA:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    return running_loss / max(1, total), 100.0 * correct / max(1, total)

@torch.inference_mode()
def evaluate(model, loader, criterion_for_loss):
    model.eval()
    total, correct = 0, 0
    running_loss = 0.0
    all_pred, all_true = [], []

    for x, y in tqdm(loader, desc="EVAL", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with autocast_or_noop():
            logits = model(x)
            loss = criterion_for_loss(logits, y)

        bs = y.size(0)
        running_loss += loss.item() * bs

        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += bs

        all_pred.append(pred.detach().cpu().numpy())
        all_true.append(y.detach().cpu().numpy())

    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)
    cm = confusion_matrix_np(all_true, all_pred, Config.NUM_CLASSES)
    f1_macro, f1_weighted = f1_from_cm(cm)

    return (
        running_loss / max(1, total),
        100.0 * correct / max(1, total),
        f1_macro,
        f1_weighted,
        cm
    )

# -------------------- Plot helpers --------------------
def plot_curves(history, out_path):
    epochs = [h["epoch"] for h in history]
    tr_loss = [h["train_loss"] for h in history]
    va_loss = [h["val_loss"] for h in history]
    tr_acc = [h["train_acc"] for h in history]
    va_acc = [h["val_acc"] for h in history]
    va_f1w = [h["val_f1_weighted"] for h in history]

    plt.figure()
    plt.plot(epochs, tr_loss, label="train_loss")
    plt.plot(epochs, va_loss, label="val_loss")
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Loss Curves")
    plt.tight_layout()
    plt.savefig(out_path / "loss_curves.png", dpi=160)
    plt.close()

    plt.figure()
    plt.plot(epochs, tr_acc, label="train_acc")
    plt.plot(epochs, va_acc, label="val_acc")
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("acc (%)")
    plt.title("Accuracy Curves")
    plt.tight_layout()
    plt.savefig(out_path / "acc_curves.png", dpi=160)
    plt.close()

    plt.figure()
    plt.plot(epochs, va_f1w, label="val_f1_weighted")
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("F1")
    plt.title("Validation Weighted F1")
    plt.tight_layout()
    plt.savefig(out_path / "val_f1_weighted_curve.png", dpi=160)
    plt.close()

def plot_confusion_matrix(cm, class_names, out_file, normalize=True):
    cm_show = cm.astype(np.float64)
    if normalize:
        row_sum = cm_show.sum(axis=1, keepdims=True) + 1e-12
        cm_show = cm_show / row_sum

    n = cm.shape[0]
    plt.figure(figsize=(11, 9))
    plt.imshow(cm_show, interpolation="nearest")
    plt.title("Confusion Matrix" + (" (normalized)" if normalize else ""))
    plt.colorbar()
    ticks = np.arange(n)
    plt.xticks(ticks, class_names, rotation=90, fontsize=7)
    plt.yticks(ticks, class_names, fontsize=7)
    plt.tight_layout()
    plt.savefig(out_file, dpi=180)
    plt.close()

def denorm(x):
    x = x.copy()
    for c in range(3):
        x[c] = x[c] * STD[c] + MEAN[c]
    return np.clip(x, 0, 1)

@torch.inference_mode()
def visualize_batch(loader, out_file, max_images=12):
    x, y = next(iter(loader))
    x = x[:max_images].cpu().numpy()
    y = y[:max_images].cpu().numpy()

    n = len(x)
    cols = 4
    rows = int(math.ceil(n / cols))
    plt.figure(figsize=(12, 3 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        img = denorm(x[i])
        plt.imshow(np.transpose(img, (1, 2, 0)))
        plt.title(class_names[int(y[i])], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.close()

# -------------------- Stage runner --------------------
history = []
best_metric = -1.0
best_val_loss = float("inf")
best_state = None
best_epoch = -1
global_epoch = 0

def run_stage(
    stage_id,
    train_loader,
    val_loader,
    optimizer_builder,
    epochs,
    train_criterion,
    use_mixup,
    min_delta_f1,
    patience,
    min_epochs,
    set_input_size_to=None,
):
    global model, best_metric, best_val_loss, best_state, best_epoch, global_epoch, history

    if set_input_size_to is not None:
        if hasattr(model, "set_input_size"):
            try:
                model.set_input_size(img_size=set_input_size_to)
            except Exception:
                pass
        if hasattr(model, "patch_embed"):
            if hasattr(model.patch_embed, "strict_img_size"):
                model.patch_embed.strict_img_size = False
            if hasattr(model.patch_embed, "img_size"):
                model.patch_embed.img_size = None

    set_trainable_stage(model, stage_id)
    optimizer = optimizer_builder(model)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=0.0)

    mix = mixup_fn if use_mixup else None
    no_improve = 0

    for local_epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, train_criterion, mixup=mix)
        va_loss, va_acc, va_f1m, va_f1w, va_cm = evaluate(model, val_loader, ce_eval)

        global_epoch += 1
        lrs = [pg["lr"] for pg in optimizer.param_groups]
        history.append({
            "epoch": global_epoch,
            "stage": stage_id,
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "val_loss": float(va_loss),
            "val_acc": float(va_acc),
            "val_f1_macro": float(va_f1m),
            "val_f1_weighted": float(va_f1w),
            "lrs": lrs,
        })

        print(
            f"Epoch {global_epoch:03d} | Stage {stage_id} | "
            f"train {tr_loss:.4f}/{tr_acc:.2f}% | "
            f"val {va_loss:.4f}/{va_acc:.2f}% | "
            f"F1w {va_f1w:.4f} | F1m {va_f1m:.4f} | "
            f"lrs {[f'{x:.2e}' for x in lrs]}"
        )

        improved = (va_f1w > best_metric + min_delta_f1) or (
            abs(va_f1w - best_metric) <= 1e-12 and va_loss < best_val_loss
        )

        if improved:
            best_metric = va_f1w
            best_val_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = global_epoch
            no_improve = 0
            print(f" -> BEST updated: val_f1_weighted={best_metric:.4f}, val_loss={best_val_loss:.4f} @ epoch {best_epoch}")
        else:
            no_improve += 1
            print(f" -> no improve ({no_improve}/{patience})")

        scheduler.step()

        if local_epoch + 1 >= min_epochs and no_improve >= patience:
            print(f"[Stage {stage_id}] Early stop at local epoch {local_epoch + 1}")
            break

# -------------------- Run stages --------------------
run_stage(
    stage_id=1,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=lambda m: build_optimizer_stage1(m),
    epochs=Config.STAGE1_EPOCHS,
    train_criterion=ce_soft,
    use_mixup=True,
    min_delta_f1=1e-6,
    patience=999,
    min_epochs=999,
    set_input_size_to=Config.IMG_SIZE_STAGE12
)

run_stage(
    stage_id=2,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=lambda m: build_optimizer_stage2(m),
    epochs=Config.STAGE2_MAX_EPOCHS,
    train_criterion=balanced_loss_stage2,
    use_mixup=False,
    min_delta_f1=Config.MIN_DELTA_F1_STAGE2,
    patience=Config.PATIENCE_STAGE2,
    min_epochs=Config.MIN_EPOCHS_STAGE2,
    set_input_size_to=Config.IMG_SIZE_STAGE12
)

train_loader_256, val_loader_256, test_loader_256 = build_loaders(Config.IMG_SIZE_STAGE3)

if best_state is not None:
    model.load_state_dict(best_state)

run_stage(
    stage_id=3,
    train_loader=train_loader_256,
    val_loader=val_loader_256,
    optimizer_builder=lambda m: build_optimizer_stage3(m),
    epochs=Config.STAGE3_EPOCHS,
    train_criterion=balanced_loss_stage3,
    use_mixup=False,
    min_delta_f1=Config.MIN_DELTA_F1_STAGE3,
    patience=Config.PATIENCE_STAGE3,
    min_epochs=Config.MIN_EPOCHS_STAGE3,
    set_input_size_to=Config.IMG_SIZE_STAGE3
)

if best_state is not None:
    model.load_state_dict(best_state)

# -------------------- Final eval on TEST with mild TTA --------------------
@torch.inference_mode()
def predict_tta_with_details(model, loader):
    model.eval()
    all_pred, all_true, all_conf = [], [], []

    for x, y in tqdm(loader, desc="TEST(TTA)", leave=False):
        x = x.to(DEVICE, non_blocking=True)

        with autocast_or_noop():
            l0 = model(x)
            l1 = model(torch.flip(x, dims=[3]))
            logits = (l0 + l1) / 2.0
            probs = torch.softmax(logits, dim=1)

        conf, pred = probs.max(dim=1)

        all_pred.append(pred.cpu().numpy())
        all_true.append(y.numpy())
        all_conf.append(conf.cpu().numpy())

    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)
    all_conf = np.concatenate(all_conf)

    cm = confusion_matrix_np(all_true, all_pred, Config.NUM_CLASSES)
    acc = (all_true == all_pred).mean() * 100.0
    f1m, f1w = f1_from_cm(cm)

    return acc, f1m, f1w, cm, all_true, all_pred, all_conf

test_acc, test_f1m, test_f1w, test_cm, test_true_all, test_pred_all, test_conf_all = predict_tta_with_details(
    model, test_loader_256
)

print("\n" + "=" * 70)
print("BEST EPOCH:", best_epoch)
print(f"BEST VAL F1w: {best_metric:.4f}")
print(f"BEST VAL LOSS: {best_val_loss:.4f}")
print(f"TEST ACC (TTA): {test_acc:.2f}% | F1w: {test_f1w:.4f} | F1m: {test_f1m:.4f}")
print("=" * 70)

# -------------------- Random 10 failed test examples --------------------
def save_random_failures(
    paths,
    y_true,
    y_pred,
    y_conf,
    class_names,
    out_dir,
    n=10,
    seed=42
):
    fail_idx = np.where(y_true != y_pred)[0]
    fail_dir = out_dir / "random_10_failed_test_examples"
    fail_dir.mkdir(parents=True, exist_ok=True)

    if len(fail_idx) == 0:
        print("No misclassified test examples found.")
        return []

    rng = np.random.default_rng(seed)
    chosen = rng.choice(fail_idx, size=min(n, len(fail_idx)), replace=False)

    meta = []
    for rank, idx in enumerate(chosen, start=1):
        src = Path(paths[idx])
        safe_true = class_names[int(y_true[idx])].replace("/", "_")
        safe_pred = class_names[int(y_pred[idx])].replace("/", "_")
        dst_name = f"{rank:02d}__true_{safe_true}__pred_{safe_pred}{src.suffix}"
        dst_path = fail_dir / dst_name
        shutil.copy2(src, dst_path)

        meta.append({
            "rank": int(rank),
            "src_path": str(src),
            "saved_name": dst_name,
            "true_idx": int(y_true[idx]),
            "pred_idx": int(y_pred[idx]),
            "true_class": class_names[int(y_true[idx])],
            "pred_class": class_names[int(y_pred[idx])],
            "confidence": float(y_conf[idx]),
        })

    with open(fail_dir / "metadata.json", "w") as fp:
        json.dump(meta, fp, indent=2)

    cols = 2
    rows = int(math.ceil(len(chosen) / cols))
    plt.figure(figsize=(12, 4 * rows))
    for i, idx in enumerate(chosen):
        img = Image.open(paths[idx]).convert("RGB")
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(
            f"true: {class_names[int(y_true[idx])]}\n"
            f"pred: {class_names[int(y_pred[idx])]}\n"
            f"conf: {y_conf[idx]:.3f}",
            fontsize=9
        )
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_dir / "random_10_failed_test_examples_grid.png", dpi=180)
    plt.close()

    return meta

# -------------------- Visualizations --------------------
Config.OUT_DIR.mkdir(parents=True, exist_ok=True)

visualize_batch(train_loader_224, Config.OUT_DIR / "augmented_batch_stage12.png", max_images=12)
plot_curves(history, Config.OUT_DIR)

_, _, _, _, val_cm = evaluate(model, val_loader_256, ce_eval)
plot_confusion_matrix(val_cm, class_names, Config.OUT_DIR / "confusion_val_norm.png", normalize=True)
plot_confusion_matrix(test_cm, class_names, Config.OUT_DIR / "confusion_test_norm.png", normalize=True)
plot_confusion_matrix(test_cm, class_names, Config.OUT_DIR / "confusion_test_raw.png", normalize=False)

failure_meta = save_random_failures(
    test_paths,
    test_true_all,
    test_pred_all,
    test_conf_all,
    class_names,
    Config.OUT_DIR,
    n=10,
    seed=42
)

print("Saved plots to:", str(Config.OUT_DIR))

# -------------------- Save model + metadata --------------------
ckpt_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_best.pth"
payload = {
    "model_name": MODEL_NAME,
    "num_classes": Config.NUM_CLASSES,
    "class_names": class_names,
    "label_map": label_map,
    "best_epoch": best_epoch,
    "best_val_f1_weighted": float(best_metric),
    "best_val_loss": float(best_val_loss),
    "history": history,
    "state_dict": model.state_dict(),
    "img_size_stage12": Config.IMG_SIZE_STAGE12,
    "img_size_stage3": Config.IMG_SIZE_STAGE3,
    "preprocessing": [
        "ShadesOfGrayColorConstancy",
        "CLAHE_LAB",
        "UnsharpMaskRGB"
    ],
}
torch.save(payload, ckpt_path)
print("Checkpoint saved:", ckpt_path)

# -------------------- Summary files --------------------
summary = {
    "model_name": MODEL_NAME,
    "best_epoch": best_epoch,
    "best_val_f1_weighted": float(best_metric),
    "best_val_loss": float(best_val_loss),
    "test_acc_tta": float(test_acc),
    "test_f1_weighted": float(test_f1w),
    "test_f1_macro": float(test_f1m),
    "num_failed_test_examples": int((test_true_all != test_pred_all).sum()),
}
summary_path = Config.OUT_DIR / "summary.json"
with open(summary_path, "w") as fp:
    json.dump(summary, fp, indent=2)

history_path = Config.OUT_DIR / "history.json"
with open(history_path, "w") as fp:
    json.dump(history, fp, indent=2)

# -------------------- Zip everything for Kaggle download --------------------
zip_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in [ckpt_path, summary_path, history_path]:
        if f.exists():
            z.write(f, arcname=f.name)

    plot_files = [
        "loss_curves.png",
        "acc_curves.png",
        "val_f1_weighted_curve.png",
        "confusion_val_norm.png",
        "confusion_test_norm.png",
        "confusion_test_raw.png",
        "augmented_batch_stage12.png",
        "random_10_failed_test_examples_grid.png",
    ]
    for fname in plot_files:
        f = Config.OUT_DIR / fname
        if f.exists():
            z.write(f, arcname=f.name)

    fail_dir = Config.OUT_DIR / "random_10_failed_test_examples"
    if fail_dir.exists():
        for root, _, files in os.walk(fail_dir):
            for file in files:
                full_path = Path(root) / file
                arcname = str(full_path.relative_to(Config.OUT_DIR))
                z.write(full_path, arcname=arcname)

print("ZIP saved:", zip_path)
print("\nKaggle Output panelinden indir:", zip_path.name)

gc.collect()
if USE_CUDA:
    torch.cuda.empty_cache()

Device: cuda | torch: 2.10.0+cu128 | timm: 1.0.22
NUM_CLASSES: 23 | train: 15557 | test: 4002
split -> train: 14011 | val: 1546
train class count min/max: 191 1265
Chosen model: swinv2_small_window8_256.ms_in1k


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

[Stage 1] Trainable: 0.06M / 48.98M


Epoch 001 | Stage 1 | train 2.4846/29.18% | val 2.2518/33.70% | F1w 0.3185 | F1m 0.2616 | lrs ['2.00e-05', '7.00e-04']
 -> BEST updated: val_f1_weighted=0.3185, val_loss=2.2518 @ epoch 1


Epoch 002 | Stage 1 | train 2.2355/37.15% | val 2.1667/35.12% | F1w 0.3327 | F1m 0.2785 | lrs ['1.81e-05', '6.33e-04']
 -> BEST updated: val_f1_weighted=0.3327, val_loss=2.1667 @ epoch 2


...

Epoch 021 | Stage 3 | train 0.0811/96.96% | val 1.6330/66.62% | F1w 0.6680 | F1m 0.6341 | lrs ['1.20e-05', '2.50e-05']
 -> no improve (1/3)


Epoch 022 | Stage 3 | train 0.0743/97.16% | val 1.6497/67.53% | F1w 0.6749 | F1m 0.6370 | lrs ['1.09e-05', '2.26e-05']
 -> no improve (2/3)


Epoch 023 | Stage 3 | train 0.0656/97.34% | val 1.7886/67.27% | F1w 0.6718 | F1m 0.6408 | lrs ['7.85e-06', '1.64e-05']
 -> no improve (3/3)
[Stage 3] Early stop at local epoch 3



BEST EPOCH: 19
BEST VAL F1w: 0.6802
BEST VAL LOSS: 1.2487
TEST ACC (TTA): 70.19% | F1w: 0.7039 | F1m: 0.6616


Saved plots to: /kaggle/working
Checkpoint saved: /kaggle/working/dermnet_swin_balanced_best.pth
ZIP saved: /kaggle/working/dermnet_swin_balanced_outputs.zip

Kaggle Output panelinden indir: dermnet_swin_balanced_outputs.zip
